# Decision Consistency Evaluation — Intel Image Classification (Scene)

Dataset: 6 natural scene classes (buildings, forest, glacier, mountain, sea, street), 150x150 real-world photography.
ImageNet-pretrained models evaluated on a different domain.
Models with high CS but low accuracy reveal the Consistently Wrong failure mode (CAG < 0).

**Update paths before running.**

In [ ]:
INTEL_TEST_DIR = r'C:\Users\Acer\Downloads\Intel Image Classification\seg_test\seg_test'
INTEL_TRAIN_DIR = r'C:\Users\Acer\Downloads\Intel Image Classification\seg_train\seg_train'
OUTPUT_DIR     = r'E:\decision_consistency_intel_outputs'
NUM_SAMPLES = 1000
SEED        = 42
ALPHA       = 0.5

In [ ]:
import os, json, random, io, csv, gc
from collections import defaultdict
import numpy as np
from scipy.stats import wilcoxon, pearsonr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models, transforms, datasets
from PIL import Image
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF']='max_split_size_mb:128'
for sub in ['tables','figures/consistency','figures/gradcam','figures/perclass','figures/ablation']:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
raw_transform=transforms.Compose([transforms.Resize((150,150)),transforms.ToTensor()])
intel_dataset=datasets.ImageFolder(root=INTEL_TEST_DIR,transform=raw_transform)
INTEL_CLASSES=intel_dataset.classes
print('Classes:',INTEL_CLASSES,'| Total:',len(intel_dataset))
all_indices=list(range(len(intel_dataset))); random.seed(SEED); random.shuffle(all_indices); selected=all_indices[:NUM_SAMPLES]
X_intel,y_intel=[],[]
for idx in tqdm(selected,desc='Loading Intel'):
    img_tensor,label=intel_dataset[idx]; X_intel.append(img_tensor); y_intel.append(label)
y_intel=np.array(y_intel); print(f'Loaded {len(X_intel)} images')

In [ ]:
zoo=[('ResNet-18',models.resnet18),('ResNet-50',models.resnet50),('VGG-16',models.vgg16),('MobileNetV2',models.mobilenet_v2)]
MODELS={}
for name,fn in zoo: print(f'Loading {name}...',end=' ',flush=True); MODELS[name]=fn(pretrained=True).eval().to(device); print('OK')
model_names=list(MODELS.keys())

In [ ]:
preprocess=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
def prepare(t): return preprocess(transforms.ToPILImage()(t.cpu().clamp(0,1))).unsqueeze(0).to(device)
def apply_jpeg(t,q=75):
    buf=io.BytesIO(); transforms.ToPILImage()(t.cpu().clamp(0,1)).save(buf,format='JPEG',quality=q); buf.seek(0); return transforms.ToTensor()(Image.open(buf))
def generate_perturbations(t):
    noise=0.01*torch.randn_like(t)
    return {'rotation':TF.rotate(t,angle=2),'brightness':TF.adjust_brightness(t,brightness_factor=1.1),'noise':torch.clamp(t+noise,0,1),'contrast':TF.adjust_contrast(t,contrast_factor=1.1),'blur':TF.gaussian_blur(t,kernel_size=3,sigma=0.5),'jpeg':apply_jpeg(t)}
def infer(model,t):
    with torch.no_grad(): probs=F.softmax(model(prepare(t)),dim=1); return probs.argmax(1).item(),probs.max().item()
def compute_metrics(op,oc,pr,alpha=ALPHA):
    K=len(pr); S=sum(1 for p,_ in pr.values() if p==op)/K; D=sum(abs(c-oc)/max(oc,1-oc) for _,c in pr.values())/K
    return {'Label_Stability':S,'Confidence_Deviation':D,'Consistency_Score':alpha*S+(1-alpha)*(1-D)}
print('Pipeline ready.')

In [ ]:
results={'Intel-Scene':{}}
for model_name,model in MODELS.items():
    per_sample=[]
    for img,true_label in tqdm(zip(X_intel,y_intel),total=len(X_intel),desc=f'Intel/{model_name}'):
        op,oc=infer(model,img); pr={k:infer(model,v) for k,v in generate_perturbations(img).items()}
        m=compute_metrics(op,oc,pr); m['true_label']=int(true_label); m['class_name']=INTEL_CLASSES[int(true_label)]; m['orig_pred']=op; m['orig_conf']=oc
        per_sample.append(m)
    results['Intel-Scene'][model_name]=per_sample
    ls=np.mean([m['Label_Stability'] for m in per_sample]); cd=np.mean([m['Confidence_Deviation'] for m in per_sample]); cs=np.mean([m['Consistency_Score'] for m in per_sample])
    print(f'  [Intel-Scene][{model_name}]  LS={ls:.3f}  CD={cd:.3f}  CS={cs:.3f}')
print('Done.')

In [ ]:
header=['Dataset','Model','Avg_Label_Stability','Avg_Confidence_Deviation','Avg_Consistency_Score','Std_Consistency_Score']
rows=[]
for mn in model_names:
    ms=results['Intel-Scene'][mn]; ls=np.mean([m['Label_Stability'] for m in ms]); cd=np.mean([m['Confidence_Deviation'] for m in ms]); cs=np.mean([m['Consistency_Score'] for m in ms]); std=np.std([m['Consistency_Score'] for m in ms])
    rows.append({'Dataset':'Intel-Scene','Model':mn,'Avg_Label_Stability':round(ls,4),'Avg_Confidence_Deviation':round(cd,4),'Avg_Consistency_Score':round(cs,4),'Std_Consistency_Score':round(std,4)})
    print(f'{mn:<14} LS={ls:.3f} CD={cd:.3f} CS={cs:.3f}')
csv_path=os.path.join(OUTPUT_DIR,'tables','summary_table_intel.csv')
with open(csv_path,'w',newline='') as f: w=csv.DictWriter(f,fieldnames=header); w.writeheader(); w.writerows(rows)
print('Saved:',csv_path)

In [ ]:
alpha_values=[0.3,0.4,0.5,0.6,0.7]
fig,ax=plt.subplots(figsize=(8,4))
for mn,marker in zip(model_names,['o','s','^','D']):
    ms=results['Intel-Scene'][mn]; ls_a=np.array([m['Label_Stability'] for m in ms]); cd_a=np.array([m['Confidence_Deviation'] for m in ms])
    vals=[float(np.mean(a*ls_a+(1-a)*(1-cd_a))) for a in alpha_values]; ax.plot(alpha_values,vals,marker=marker,label=mn,linewidth=2,markersize=7)
ax.set_xlabel('alpha'); ax.set_ylabel('Avg Consistency Score'); ax.set_title('Alpha-Ablation Intel-Scene'); ax.set_xticks(alpha_values); ax.set_ylim(0,1); ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
path=os.path.join(OUTPUT_DIR,'figures','ablation','alpha_ablation_intel.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)

In [ ]:
cs_arrays={mn:np.array([m['Consistency_Score'] for m in results['Intel-Scene'][mn]]) for mn in model_names}
print('=== Wilcoxon Tests (Intel-Scene) ===')
for i,mn1 in enumerate(model_names):
    for mn2 in model_names[i+1:]:
        w,p=wilcoxon(cs_arrays[mn1],cs_arrays[mn2]); print(f'  {mn1} vs {mn2}: W={w:.1f} p={p:.4f} {"***" if p<0.05 else "ns"}')

In [ ]:
# CAG Analysis: Intel-Scene models are Consistently Wrong (CAG < 0)
# High CS with low accuracy reveals systematic cross-domain failure
print('CAG Analysis (Intel-Scene):')
print(f'{"Model":<14} {"CS":>7}  Note')
print('-'*45)
for mn in model_names:
    cs=np.mean([m['Consistency_Score'] for m in results['Intel-Scene'][mn]])
    print(f'{mn:<14} {cs:>7.4f}  -> run notebook 12 for corrected CAG with linear probe accuracy')